In [15]:
import os
import cv2
from pathlib import Path
from ultralytics import YOLO


In [17]:
# Load YOLO model
model = YOLO("yolo11s.pt")  

In [19]:
# Path to traffic images
image_folder = "./mbjb_traffic_images"
output_folder = "./mbjb_traffic_images/cropped_day_and_night" 

In [21]:
# Classes based on YOLO model
# Common YOLO classes: 2=car, 3=motorcycle, 5=bus, 7=truck
class_map = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck",
}

In [23]:
# Create output folders if they don't exist
for cls_name in class_map.values():
    os.makedirs(os.path.join(output_folder, cls_name), exist_ok=True)

In [27]:
confidence_threshold = 0.6  

for img_file in Path(image_folder).glob("*.jpg"):
    img = cv2.imread(str(img_file))
    
    # Skip if image failed to load
    if img is None:
        print(f"Warning: Failed to load image {img_file.name}, skipping...")
        continue

    results = model(img, imgsz=1280)[0]

    for i, box in enumerate(results.boxes):
        # Skip low-confidence detections
        conf = float(box.conf)
        if conf < confidence_threshold:
            continue

        cls_id = int(box.cls)
        if cls_id not in class_map:
            continue

        # Get bounding box and crop the vehicle
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        crop = img[y1:y2, x1:x2]
        #crop = cv2.resize(crop, (224, 224), interpolation=cv2.INTER_CUBIC)

        # Skip empty or invalid crops
        if crop.size == 0:
            print(f"Warning: Empty crop in {img_file.name}, skipping...")
            continue

        cls_name = class_map[cls_id]

        # Save cropped image
        save_path = os.path.join(output_folder, cls_name, f"{img_file.stem}_{i}.jpg")
        cv2.imwrite(save_path, crop)

    print(f"Processed: {img_file.name}")

print("Cropping complete.")


0: 736x1280 2 persons, 28 cars, 1 motorcycle, 2 buss, 5 trucks, 555.3ms
Speed: 7.4ms preprocess, 555.3ms inference, 2.1ms postprocess per image at shape (1, 3, 736, 1280)
Processed: L01_Cam1_2025-05-24_16-48-03.jpg

0: 736x1280 30 cars, 1 bus, 5 trucks, 562.4ms
Speed: 7.6ms preprocess, 562.4ms inference, 2.6ms postprocess per image at shape (1, 3, 736, 1280)
Processed: L01_Cam2_2025-05-24_16-54-12.jpg

0: 736x1280 2 persons, 29 cars, 4 motorcycles, 4 trucks, 556.4ms
Speed: 10.1ms preprocess, 556.4ms inference, 2.2ms postprocess per image at shape (1, 3, 736, 1280)
Processed: L02_Cam1_2025-05-24_16-58-04.jpg

0: 736x1280 20 cars, 3 airplanes, 2 buss, 5 trucks, 953.3ms
Speed: 10.5ms preprocess, 953.3ms inference, 2.7ms postprocess per image at shape (1, 3, 736, 1280)
Processed: L02_Cam2_2025-05-24_16-58-15.jpg

0: 736x1280 4 persons, 24 cars, 3 trucks, 756.9ms
Speed: 9.4ms preprocess, 756.9ms inference, 2.1ms postprocess per image at shape (1, 3, 736, 1280)
Processed: L03_Cam1_2025-05-2